# Fama-French SMB & HML Factor Replication

This notebook replicates the **SMB** (Small Minus Big) and **HML** (High Minus Low)
factors from:

> Fama, E.F. and French, K.R. (1993), "Common Risk Factors in the Returns on Stocks
> and Bonds", *Journal of Financial Economics*, 33(1), 3-56.

We build the factors from **raw WRDS data** (Compustat, CRSP, and the CCM link table)
and compare our results against the **official Ken French factors** from his data library.

## Why these factors matter

The Fama-French three-factor model extended the CAPM by adding two empirically
motivated risk factors:

- **SMB** captures the *size premium*: historically, small-cap stocks earned higher
  average returns than large-cap stocks.
- **HML** captures the *value premium*: historically, high book-to-market ("value")
  stocks earned higher average returns than low book-to-market ("growth") stocks.

By constructing these factors ourselves, we gain deep understanding of the
methodological choices Fama and French made and why each step matters.

## 0. Configuration

Set your preferences here. You can either:
1. **Query WRDS live** by setting `DATA_SOURCE = 'wrds'` (requires a WRDS account), or
2. **Use local files** by setting `DATA_SOURCE = 'local'` and providing file paths.

The `START_DATE` controls how far back the sample goes. Fama and French (1993) used
data from 1963 onward, but we default to 2000 for faster computation. Change it as needed.

In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────────

DATA_SOURCE = 'wrds'   # 'wrds' to query live, 'local' to use saved files
START_DATE  = '2000-01-01'  # Change to '1963-07-01' for full FF sample

# If DATA_SOURCE == 'local', set these paths to your saved Parquet/CSV files:
LOCAL_COMPUSTAT_PATH = 'data/compustat_data.parquet'
LOCAL_CRSP_PATH      = 'data/crsp_data.parquet'
LOCAL_CCM_PATH       = 'data/ccm_data.parquet'

# WRDS credentials (only needed if DATA_SOURCE == 'wrds')
# The wrds library will prompt for username/password if not configured.
# Alternatively, set up a ~/.pgpass file per WRDS documentation.
WRDS_USERNAME = None  # e.g. 'your_wrds_id'

# ── END CONFIGURATION ──────────────────────────────────────────────

## 1. Imports

We use standard Python data-science libraries plus `wrds` for database access
and `pandas_datareader` to download official Ken French factors for comparison.

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
from dateutil.relativedelta import relativedelta
from pandas.tseries.offsets import MonthEnd, YearEnd
from scipy import stats
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print('All imports successful.')

## 2. Helper Functions

All custom logic lives inside this notebook. We define reusable helpers here
so they can be tested independently.

In [ ]:
# ── Helper: value-weighted average ────────────────────────────────
def wavg(group, avg_name, weight_name):
    """
    Compute the weighted average of `avg_name` using `weight_name` as weights.
    Used to compute value-weighted portfolio returns.

    Why value-weighting?  Fama and French (1993) value-weight portfolio returns
    so that each stock's contribution reflects its economic importance (market cap).
    Equal weighting would give outsized influence to tiny, illiquid stocks.
    """
    d = group[avg_name]
    w = group[weight_name]
    try:
        return (d * w).sum() / w.sum()
    except ZeroDivisionError:
        return np.nan


# ── Helper: size bucket ───────────────────────────────────────────
def sz_bucket(row):
    """
    Assign a stock to 'S' (Small) or 'B' (Big) based on whether its June
    market equity is below or above the NYSE median.

    Why the NYSE median?  Fama and French use NYSE-only breakpoints to avoid
    letting the huge number of small NASDAQ/AMEX stocks push the median down,
    which would put almost every stock into the 'Big' bucket and defeat the
    purpose of the size sort.
    """
    if pd.isna(row['me']) or pd.isna(row['sizemedn']):
        return ''
    elif row['me'] <= row['sizemedn']:
        return 'S'
    else:
        return 'B'


# ── Helper: book-to-market bucket ────────────────────────────────
def bm_bucket(row):
    """
    Assign a stock to 'L' (Low/Growth), 'M' (Medium), or 'H' (High/Value)
    based on where its book-to-market falls relative to NYSE 30th/70th
    percentiles.

    Why 30/70 percentiles?  Fama and French (1993) chose these thresholds
    to create three B/M groups of roughly equal economic weight among NYSE
    stocks, while ensuring the extreme portfolios (L and H) have enough
    spread to capture the value premium.
    """
    if pd.isna(row['beme']) or pd.isna(row['bm30']) or pd.isna(row['bm70']):
        return ''
    if 0 <= row['beme'] <= row['bm30']:
        return 'L'
    elif row['beme'] <= row['bm70']:
        return 'M'
    elif row['beme'] > row['bm70']:
        return 'H'
    else:
        return ''


# ── Helper: compute book equity from Compustat ───────────────────
def compute_book_equity(comp):
    """
    Build book equity using the Fama-French preferred-stock fallback hierarchy:
      ps = pstkrv (redemption value)
           -> pstkl  (liquidating value)   if pstkrv is missing
           -> pstk   (par value)           if both are missing
           -> 0                            if all are missing

      BE = stockholders' equity (seq) + deferred taxes (txditc) - preferred stock (ps)

    Only positive BE is kept, since negative book equity has no clear economic
    meaning for the B/M ratio.
    """
    df = comp.copy()
    df['ps'] = np.where(df['pstkrv'].isnull(), df['pstkl'], df['pstkrv'])
    df['ps'] = np.where(df['ps'].isnull(), df['pstk'], df['ps'])
    df['ps'] = np.where(df['ps'].isnull(), 0, df['ps'])
    df['txditc'] = df['txditc'].fillna(0)
    df['be'] = (df['seq'] + df['txditc'] - df['ps']).astype(float)
    df['be'] = np.where(df['be'] > 0, df['be'], np.nan)
    return df


# ── Helper: compute adjusted returns with delisting ──────────────
def compute_retadj(ret, dlret):
    """
    Combine holding-period return with delisting return:
      retadj = (1 + ret) * (1 + dlret) - 1

    Why delisting returns matter:  When a stock is delisted (e.g. due to
    bankruptcy or acquisition), its final return is recorded separately.
    Ignoring delisting returns creates survivorship bias - we'd miss the
    large negative returns from firms that went bankrupt, biasing average
    returns upward.
    """
    ret = ret if not pd.isna(ret) else 0.0
    dlret = dlret if not pd.isna(dlret) else 0.0
    return (1 + ret) * (1 + dlret) - 1


# ── Helper: SMB/HML formulas ─────────────────────────────────────
def compute_smb_hml(ff):
    """
    Compute SMB and HML from the 2x3 portfolio returns:

      SMB = (SL + SM + SH)/3  -  (BL + BM + BH)/3
      HML = (SH + BH)/2       -  (SL + BL)/2

    SMB is the average return on small portfolios minus the average return on
    big portfolios, controlling for B/M.
    HML is the average return on high-B/M portfolios minus the average return
    on low-B/M portfolios, controlling for size.
    """
    ff = ff.copy()
    ff['WSMB'] = (ff['SL'] + ff['SM'] + ff['SH']) / 3 - (ff['BL'] + ff['BM'] + ff['BH']) / 3
    ff['WHML'] = (ff['SH'] + ff['BH']) / 2 - (ff['SL'] + ff['BL']) / 2
    return ff


print('Helper functions defined.')

## 3. Load Raw Data

We need three raw datasets from WRDS:

| Dataset | WRDS Table | Purpose |
|---------|-----------|--------|
| **Compustat** | `comp.funda` | Annual accounting data (book equity) |
| **CRSP** | `crsp.msf`, `crsp.msenames`, `crsp.msedelist` | Monthly stock returns, exchange codes, delisting returns |
| **CCM** | `crsp.ccmxpf_linktable` | Links Compustat GVKEYs to CRSP PERMNOs |

### Why do we need all three?

- **Compustat** gives us book equity (the numerator of B/M).
- **CRSP** gives us stock prices, returns, and shares outstanding - needed for
  market equity (the denominator of B/M) and for computing portfolio returns.
- **CCM link table** maps between the two databases. Compustat identifies firms
  by `gvkey`; CRSP identifies securities by `permno`. The link table provides the
  bridge, including the date range over which each link is valid.

In [ ]:
if DATA_SOURCE == 'wrds':
    import wrds
    conn = wrds.Connection(wrds_username=WRDS_USERNAME) if WRDS_USERNAME else wrds.Connection()

    # ── Compustat ──
    comp = conn.raw_sql(f"""
        SELECT gvkey, datadate, at, pstkl, txditc, pstkrv, seq, pstk
        FROM comp.funda
        WHERE indfmt='INDL'
          AND datafmt='STD'
          AND popsrc='D'
          AND consol='C'
          AND datadate >= '{START_DATE}'
    """, date_cols=['datadate'])

    # ── CRSP monthly stock file + name history ──
    crsp_m = conn.raw_sql(f"""
        SELECT a.permno, a.permco, a.date, b.shrcd, b.exchcd,
               a.ret, a.retx, a.shrout, a.prc
        FROM crsp.msf AS a
        LEFT JOIN crsp.msenames AS b
          ON a.permno = b.permno
         AND b.namedt <= a.date
         AND a.date <= b.nameendt
        WHERE a.date > '{START_DATE}'
          AND b.exchcd BETWEEN 1 AND 3
    """, date_cols=['date'])

    # ── CRSP delisting returns ──
    dlret = conn.raw_sql("""
        SELECT permno, dlret, dlstdt
        FROM crsp.msedelist
    """, date_cols=['dlstdt'])

    # ── CCM link table ──
    ccm = conn.raw_sql("""
        SELECT gvkey, lpermno AS permno, linktype, linkprim,
               linkdt, linkenddt
        FROM crsp.ccmxpf_linktable
        WHERE SUBSTR(linktype,1,1) = 'L'
          AND (linkprim = 'C' OR linkprim = 'P')
    """, date_cols=['linkdt', 'linkenddt'])

    conn.close()
    print(f'Loaded from WRDS: comp={len(comp)}, crsp_m={len(crsp_m)}, dlret={len(dlret)}, ccm={len(ccm)}')

elif DATA_SOURCE == 'local':
    def _load(path):
        if path.endswith('.parquet'):
            return pd.read_parquet(path)
        else:
            return pd.read_csv(path, parse_dates=True)

    comp   = _load(LOCAL_COMPUSTAT_PATH)
    crsp_m = _load(LOCAL_CRSP_PATH)

    # Local CRSP file may already include dlret merged, or separate
    ccm = _load(LOCAL_CCM_PATH)

    # Delisting returns: check if already merged into crsp_m
    if 'dlret' not in crsp_m.columns:
        raise ValueError(
            'Local crsp_data must include dlret column, '
            'or provide a separate delisting file.'
        )
    dlret = pd.DataFrame()  # placeholder, already in crsp_m
    print(f'Loaded from local files: comp={len(comp)}, crsp_m={len(crsp_m)}, ccm={len(ccm)}')

else:
    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}. Use 'wrds' or 'local'.")

## 4. Compustat Block: Book Equity Construction

### What is book equity?

Book equity (BE) is the accounting value of shareholders' claim on the firm.
Fama and French (1993) define it as:

$$
BE = \text{Stockholders' Equity (SEQ)} + \text{Deferred Taxes (TXDITC)} - \text{Preferred Stock (PS)}
$$

### Preferred stock fallback hierarchy

Compustat reports preferred stock in three ways. Fama and French use this
priority order:

1. **Redemption value** (`pstkrv`) — what the firm would pay to retire preferred shares
2. **Liquidating value** (`pstkl`) — what preferred holders get in liquidation
3. **Par value** (`pstk`) — the nominal face value

If all are missing, preferred stock is set to zero.

### Why keep only positive book equity?

A negative book equity means liabilities exceed assets — such firms have no
meaningful book-to-market ratio, so they're excluded from the B/M sort.

### Compustat history count

We count how many years each firm has appeared in Compustat. Fama and French
require at least **two years** of Compustat data before a firm enters portfolios,
to reduce the impact of backfill bias (Compustat historically added firms with
retroactive data).

In [ ]:
# ── Compustat: build book equity ─────────────────────────────────
comp['year'] = comp['datadate'].dt.year

comp = compute_book_equity(comp)

# Count years in Compustat (for backfill-bias filter)
comp = comp.sort_values(by=['gvkey', 'datadate'])
comp['count'] = comp.groupby(['gvkey']).cumcount()

comp = comp[['gvkey', 'datadate', 'year', 'be', 'count']]

print(f'Compustat after BE construction: {len(comp)} firm-years')
print(f'Positive BE: {comp["be"].notna().sum()} | Missing/negative BE: {comp["be"].isna().sum()}')
comp.head(10)

## 5. CRSP Block: Returns, Market Equity, and Delisting Adjustment

### Why delisting returns matter

When a stock is **delisted** (e.g., bankruptcy, merger, regulatory action), CRSP
records the final return in a separate delisting file (`crsp.msedelist`). If we
ignore this, we create **survivorship bias** — we'd miss catastrophic losses from
bankrupt firms, making average returns look better than they actually were.

The adjusted return is:

$$
r_{\text{adj}} = (1 + r_{\text{ret}}) \times (1 + r_{\text{delist}}) - 1
$$

### Market equity

$$
ME = |\text{Price}| \times \text{Shares Outstanding}
$$

We take the absolute value of price because CRSP uses negative prices to
indicate bid/ask midpoints when closing prices are unavailable.

### Why permco-level aggregation matters

A single company (`permco`) can have multiple share classes, each with its own
`permno`. To get the **total** market cap of the firm, we must sum ME across
all `permno`s belonging to the same `permco`. We then assign this total ME to
the `permno` with the largest individual ME (the primary trading vehicle).

### December market equity

The book-to-market ratio uses **December** ME as the denominator. Why December?
Fama and French want the market value to be **known before** portfolios are
formed in June. Since most firms have fiscal year ends in December, the
December ME is the most recent figure that aligns with the annual book value.
Using the December ME of year $t-1$ with portfolios formed in June of year $t$
ensures a **6-month gap** that prevents look-ahead bias.

### July-to-June weight logic

Fama and French form portfolios at the end of June each year and hold them
for 12 months (July through June). The **weight** for value-weighted returns
must track each stock's evolving market cap through the year. The `mebase`
is the stock's ME at the start of the holding period, and `cumretx`
(cumulative ex-dividend return) scales it forward month by month.

In [ ]:
# ── CRSP: format columns ────────────────────────────────────────
crsp_m[['permco', 'permno', 'shrcd', 'exchcd']] = (
    crsp_m[['permco', 'permno', 'shrcd', 'exchcd']].astype(int)
)
crsp_m['jdate'] = crsp_m['date'] + MonthEnd(0)

print(f'CRSP monthly records: {len(crsp_m)}')

### 5a. Merge Delisting Returns

In [ ]:
# ── Merge delisting returns ─────────────────────────────────────
if len(dlret) > 0:  # If from WRDS (local files may already have dlret)
    dlret['permno'] = dlret['permno'].astype(int)
    dlret['jdate'] = dlret['dlstdt'] + MonthEnd(0)
    crsp = pd.merge(crsp_m, dlret[['permno', 'dlret', 'jdate']],
                    how='left', on=['permno', 'jdate'])
else:
    crsp = crsp_m.copy()

crsp['dlret'] = crsp['dlret'].fillna(0)
crsp['ret'] = crsp['ret'].fillna(0)
crsp['retadj'] = (1 + crsp['ret']) * (1 + crsp['dlret']) - 1

# Market equity
crsp['me'] = crsp['prc'].abs() * crsp['shrout']
crsp = crsp.drop(['dlret', 'prc', 'shrout'], axis=1, errors='ignore')
crsp = crsp.sort_values(by=['jdate', 'permco', 'me'])

print(f'CRSP after delisting merge: {len(crsp)} records')
print(f'Non-zero delisting adjustments: {(crsp["retadj"] != crsp["ret"]).sum()}')

### 5b. Aggregate Market Equity at permco Level

In [ ]:
# ── Permco-level aggregation ────────────────────────────────────
# Sum of ME across all permno within each permco/date
crsp_summe = crsp.groupby(['jdate', 'permco'])['me'].sum().reset_index()
# Largest ME within each permco/date (to find primary permno)
crsp_maxme = crsp.groupby(['jdate', 'permco'])['me'].max().reset_index()

# Keep the permno with the largest individual ME
crsp1 = pd.merge(crsp, crsp_maxme, how='inner', on=['jdate', 'permco', 'me'])
crsp1 = crsp1.drop(['me'], axis=1)

# Replace with total permco-level ME
crsp2 = pd.merge(crsp1, crsp_summe, how='inner', on=['jdate', 'permco'])
crsp2 = crsp2.sort_values(by=['permno', 'jdate']).drop_duplicates()

print(f'CRSP after permco aggregation: {len(crsp2)} records')

### 5c. December Market Equity and July-to-June Weights

In [ ]:
# ── December ME ─────────────────────────────────────────────────
crsp2['year'] = crsp2['jdate'].dt.year
crsp2['month'] = crsp2['jdate'].dt.month

decme = crsp2[crsp2['month'] == 12].copy()
decme = decme[['permno', 'date', 'jdate', 'me', 'year']].rename(columns={'me': 'dec_me'})

# ── July-to-June calendar ───────────────────────────────────────
crsp2['ffdate'] = crsp2['jdate'] + MonthEnd(-6)
crsp2['ffyear'] = crsp2['ffdate'].dt.year
crsp2['ffmonth'] = crsp2['ffdate'].dt.month
crsp2['1+retx'] = 1 + crsp2['retx']
crsp2 = crsp2.sort_values(by=['permno', 'date'])

# Cumulative ex-dividend return within each July-June period
crsp2['cumretx'] = crsp2.groupby(['permno', 'ffyear'])['1+retx'].cumprod()
crsp2['lcumretx'] = crsp2.groupby(['permno'])['cumretx'].shift(1)

# Lagged ME
crsp2['lme'] = crsp2.groupby(['permno'])['me'].shift(1)
crsp2['count'] = crsp2.groupby(['permno']).cumcount()
crsp2['lme'] = np.where(crsp2['count'] == 0, crsp2['me'] / crsp2['1+retx'], crsp2['lme'])

# Baseline ME (start of each July-June period)
mebase = crsp2[crsp2['ffmonth'] == 1][['permno', 'ffyear', 'lme']].rename(
    columns={'lme': 'mebase'}
)
crsp3 = pd.merge(crsp2, mebase, how='left', on=['permno', 'ffyear'])
crsp3['wt'] = np.where(
    crsp3['ffmonth'] == 1,
    crsp3['lme'],
    crsp3['mebase'] * crsp3['lcumretx']
)

# Shift Dec ME forward by one year (Dec year t-1 used for June year t sorts)
decme['year'] = decme['year'] + 1
decme = decme[['permno', 'year', 'dec_me']]

print(f'CRSP with weights: {len(crsp3)} records')

### 5d. June Snapshot

We take a snapshot of each stock's characteristics as of **June** each year.
This is the point at which portfolios are formed. The June ME is used for the
size sort, and Dec ME of the prior year is used for the B/M sort.

In [ ]:
# ── June snapshot ───────────────────────────────────────────────
crsp3_jun = crsp3[crsp3['month'] == 6]
crsp_jun = pd.merge(crsp3_jun, decme, how='inner', on=['permno', 'year'])
crsp_jun = crsp_jun[[
    'permno', 'date', 'jdate', 'shrcd', 'exchcd',
    'retadj', 'me', 'wt', 'cumretx', 'mebase', 'lme', 'dec_me'
]]
crsp_jun = crsp_jun.sort_values(by=['permno', 'jdate']).drop_duplicates()

print(f'June snapshots: {len(crsp_jun)} stock-years')

## 6. CCM Block: Linking Compustat to CRSP

### Why is the CCM link table needed?

Compustat and CRSP are maintained by different organizations and use different
identifiers:
- Compustat uses **GVKEY** (a firm-level identifier)
- CRSP uses **PERMNO** (a security-level identifier)

The CRSP-Compustat Merged (CCM) link table provides the mapping between these
identifiers, along with the **date range** during which each link is valid.

We only use links where:
- `linktype` starts with 'L' (primary link types)
- `linkprim` is 'C' or 'P' (primary security link)

### June `jdate` for merging

We create a June `jdate` from each Compustat fiscal-year-end date:
1. Round `datadate` up to the calendar year end -> `yearend`
2. Add 6 months -> `jdate` (the June when this data would be used)

This ensures that fiscal-year data is only used **after** it would have been
publicly available.

### Book-to-market ratio

$$
B/M = \frac{BE \times 1000}{\text{Dec ME}}
$$

The factor of 1000 adjusts for Compustat reporting in millions vs. CRSP
reporting in thousands.

In [ ]:
# ── CCM: link Compustat to CRSP ─────────────────────────────────
ccm['linkenddt'] = ccm['linkenddt'].fillna(pd.to_datetime('today'))

ccm1 = pd.merge(
    comp[['gvkey', 'datadate', 'be', 'count']],
    ccm,
    how='left',
    on=['gvkey']
)

ccm1['yearend'] = ccm1['datadate'] + YearEnd(0)
ccm1['jdate'] = ccm1['yearend'] + MonthEnd(6)

# Keep only links valid at the June formation date
ccm2 = ccm1[(ccm1['jdate'] >= ccm1['linkdt']) & (ccm1['jdate'] <= ccm1['linkenddt'])]
ccm2 = ccm2[['gvkey', 'permno', 'datadate', 'yearend', 'jdate', 'be', 'count']]

# Merge with June CRSP data
ccm_jun = pd.merge(crsp_jun, ccm2, how='inner', on=['permno', 'jdate'])
ccm_jun['beme'] = ccm_jun['be'] * 1000 / ccm_jun['dec_me']

print(f'CCM merged June records: {len(ccm_jun)}')
ccm_jun[['permno', 'jdate', 'me', 'be', 'dec_me', 'beme']].head(10)

## 7. NYSE Breakpoints

### Why NYSE-only breakpoints?

Fama and French (1993) compute size and B/M breakpoints using **only NYSE stocks**.
This is a critical design choice:

- NASDAQ and AMEX have many more small, speculative firms than NYSE.
- If we used all exchanges, the size median would be dragged down, putting
  the vast majority of firms into the "Big" category.
- Using NYSE-only breakpoints ensures the **Big** and **Small** groups are
  economically meaningful: the median NYSE firm is genuinely mid-sized.

### Breakpoint definitions

- **Size**: NYSE median ME -> stocks below are "Small", above are "Big"
- **B/M**: NYSE 30th and 70th percentiles -> creates three groups:
  - **Low (Growth)**: B/M <= 30th percentile
  - **Medium**: 30th < B/M <= 70th percentile
  - **High (Value)**: B/M > 70th percentile

### Why June sorts?

Portfolios are formed at the **end of June** each year. This gives a 6-month
gap between the fiscal year end (typically December) and portfolio formation.
The gap ensures that annual accounting data (book equity) would have been
publicly available by the time investors sort stocks - avoiding **look-ahead bias**.

### Filter criteria

Following FF (1993), we require:
- NYSE stocks only (`exchcd == 1`)
- Positive book-to-market (`beme > 0`)
- Positive market equity (`me > 0`)
- At least 2 years of Compustat data (`count >= 1`, since count is 0-indexed)
- Ordinary common shares only (`shrcd` in 10, 11)

In [ ]:
# ── NYSE breakpoints ────────────────────────────────────────────
nyse = ccm_jun[
    (ccm_jun['exchcd'] == 1) &
    (ccm_jun['beme'] > 0) &
    (ccm_jun['me'] > 0) &
    (ccm_jun['count'] >= 1) &
    ((ccm_jun['shrcd'] == 10) | (ccm_jun['shrcd'] == 11))
]

# Size: NYSE median ME
nyse_sz = nyse.groupby(['jdate'])['me'].median().to_frame().reset_index().rename(
    columns={'me': 'sizemedn'}
)

# B/M: NYSE 30th and 70th percentiles
nyse_bm = nyse.groupby(['jdate'])['beme'].describe(percentiles=[0.3, 0.7]).reset_index()
nyse_bm = nyse_bm[['jdate', '30%', '70%']].rename(columns={'30%': 'bm30', '70%': 'bm70'})

nyse_breaks = pd.merge(nyse_sz, nyse_bm, how='inner', on=['jdate'])

print(f'NYSE breakpoints computed for {len(nyse_breaks)} June dates')
nyse_breaks.head()

## 8. Assign June Portfolios

Each stock is placed into one of **six portfolios** based on the intersection
of the 2 size groups x 3 B/M groups:

|  | Low B/M (Growth) | Medium B/M | High B/M (Value) |
|--|-------------------|------------|-------------------|
| **Small** | SL | SM | SH |
| **Big** | BL | BM | BH |

These portfolio assignments are made in **June** and held for the following
12 months (July through June of the next year).

In [ ]:
# ── Assign portfolios ───────────────────────────────────────────
ccm1_jun = pd.merge(ccm_jun, nyse_breaks, how='left', on=['jdate'])

# Ensure numeric types
cols = ['beme', 'me', 'count']
ccm1_jun[cols] = ccm1_jun[cols].astype('float')

# Filter mask: positive B/M, positive ME, enough Compustat history
mask = (
    (ccm1_jun['beme'] > 0) &
    (ccm1_jun['me'] > 0) &
    (ccm1_jun['count'] >= 1)
)

ccm1_jun['szport'] = ''
ccm1_jun['bmport'] = ''
ccm1_jun.loc[mask, 'szport'] = ccm1_jun.loc[mask].apply(sz_bucket, axis=1)
ccm1_jun.loc[mask, 'bmport'] = ccm1_jun.loc[mask].apply(bm_bucket, axis=1)

ccm1_jun['posbm'] = np.where(mask, 1, 0)
ccm1_jun['nonmissport'] = np.where(ccm1_jun['bmport'] != '', 1, 0)

# Store June portfolio assignments
june = ccm1_jun[['permno', 'date', 'jdate', 'bmport', 'szport', 'posbm', 'nonmissport']].copy()
june['ffyear'] = june['jdate'].dt.year

print(f'Portfolio assignments: {len(june)} stock-years')
print(june.groupby(['szport', 'bmport']).size().unstack(fill_value=0))

## 9. Merge Assignments to Monthly Returns

We merge the June portfolio labels back into the full monthly CRSP dataset.
Each stock keeps its June assignment for the 12 months from July of year $t$
through June of year $t+1$. This is achieved by matching on `permno` and `ffyear`.

In [ ]:
# ── Merge June assignments to monthly returns ───────────────────
crsp3_cols = ['date', 'permno', 'shrcd', 'exchcd', 'retadj', 'me', 'wt',
              'cumretx', 'ffyear', 'jdate']
crsp3_sub = crsp3[crsp3_cols]

ccm3 = pd.merge(
    crsp3_sub,
    june[['permno', 'ffyear', 'szport', 'bmport', 'posbm', 'nonmissport']],
    how='left',
    on=['permno', 'ffyear']
)

# Keep only valid portfolio members
ccm4 = ccm3[
    (ccm3['wt'] > 0) &
    (ccm3['posbm'] == 1) &
    (ccm3['nonmissport'] == 1) &
    ((ccm3['shrcd'] == 10) | (ccm3['shrcd'] == 11))
]

print(f'Monthly records for portfolio returns: {len(ccm4)}')

## 10. Compute Value-Weighted Portfolio Returns

### Why value weighting?

Fama and French (1993) use **value-weighted** returns for their factor portfolios.
Value weighting means each stock's return is weighted by its market cap. This
ensures:

1. The portfolio reflects the **investable opportunity set** — you could actually
   hold this portfolio in practice.
2. Tiny illiquid stocks don't dominate the returns (as they would with equal weighting).
3. The factors capture economically meaningful size and value premiums.

In [ ]:
# ── Value-weighted portfolio returns ─────────────────────────────
vwret = ccm4.groupby(['jdate', 'szport', 'bmport']).apply(
    wavg, 'retadj', 'wt', include_groups=False
).to_frame().reset_index().rename(columns={0: 'vwret'})
vwret['sbport'] = vwret['szport'] + vwret['bmport']

# Firm counts per portfolio
vwret_n = ccm4.groupby(['jdate', 'szport', 'bmport'])['retadj'].count().reset_index().rename(
    columns={'retadj': 'n_firms'}
)
vwret_n['sbport'] = vwret_n['szport'] + vwret_n['bmport']

# Pivot to wide format
ff_factors = vwret.pivot(index='jdate', columns='sbport', values='vwret').reset_index()
ff_nfirms = vwret_n.pivot(index='jdate', columns='sbport', values='n_firms').reset_index()

print(f'Portfolio returns computed for {len(ff_factors)} months')
ff_factors.head()

## 11. Construct SMB and HML

The Fama-French factors are built from the 2x3 portfolios:

$$
\text{SMB} = \frac{SL + SM + SH}{3} - \frac{BL + BM + BH}{3}
$$

$$
\text{HML} = \frac{SH + BH}{2} - \frac{SL + BL}{2}
$$

- **SMB** (Small Minus Big) averages across B/M groups to isolate the size effect.
- **HML** (High Minus Low) averages across size groups to isolate the value effect.

In [ ]:
# ── Construct SMB and HML ────────────────────────────────────────
ff_factors = compute_smb_hml(ff_factors)
ff_factors = ff_factors.rename(columns={'jdate': 'date'})

# Firm counts
ff_nfirms['TOTAL'] = (
    ff_nfirms['SL'] + ff_nfirms['SM'] + ff_nfirms['SH'] +
    ff_nfirms['BL'] + ff_nfirms['BM'] + ff_nfirms['BH']
)
ff_nfirms = ff_nfirms.rename(columns={'jdate': 'date'})

print('Our constructed factors (first 10 months):')
ff_factors[['date', 'WSMB', 'WHML']].head(10)

## 12. Compare with Official Ken French Factors

We download the official Fama-French three-factor data from
[Ken French's Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html)
and compare our SMB and HML against the official series.

We use `pandas_datareader` to fetch the data automatically.

In [ ]:
# ── Download official Ken French factors ────────────────────────
try:
    import pandas_datareader.data as web
    ff_official = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start=START_DATE)[0]
except ImportError:
    print('pandas_datareader not installed. Installing...')
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas-datareader', '-q'])
    import pandas_datareader.data as web
    ff_official = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start=START_DATE)[0]

# Ken French data is in percentage points -> convert to decimal
ff_official = ff_official / 100.0
ff_official.index = ff_official.index.to_timestamp() + MonthEnd(0)
ff_official = ff_official.reset_index().rename(columns={'index': 'date', 'Date': 'date'})
# Handle potential column name variations
if 'date' not in ff_official.columns:
    ff_official = ff_official.rename(columns={ff_official.columns[0]: 'date'})

print(f'Official FF factors: {len(ff_official)} months')
print(f'Date range: {ff_official["date"].min()} to {ff_official["date"].max()}')
ff_official.head()

In [ ]:
# ── Merge our factors with official ─────────────────────────────
ff_comp = pd.merge(
    ff_factors[['date', 'WSMB', 'WHML']].rename(columns={'WSMB': 'Our SMB', 'WHML': 'Our HML'}),
    ff_official[['date', 'SMB', 'HML']].rename(columns={'SMB': 'FF SMB', 'HML': 'FF HML'}),
    how='inner',
    on='date'
)

print(f'Comparison sample: {len(ff_comp)} overlapping months')
print(f'Date range: {ff_comp["date"].min()} to {ff_comp["date"].max()}')
ff_comp.head()

## 13. Statistical Comparison

We evaluate our replication quality using:

1. **Pearson correlation** — measures linear co-movement
2. **OLS regression (beta & R-squared)** — measures how closely our factor tracks the official one
3. **Cointegration test** — tests whether the two series share a long-run equilibrium

A good replication should have:
- Correlation > 0.95
- Beta close to 1.0
- R-squared > 0.90
- Cointegration p-value < 0.05

In [ ]:
# ── Statistical comparison ───────────────────────────────────────
def compare_factor(our_col, ff_col, name):
    """Run correlation, regression, and cointegration for one factor."""
    df = ff_comp[[our_col, ff_col]].dropna()
    y = df[our_col]
    x = df[ff_col]

    # Correlation
    corr, corr_p = stats.pearsonr(y, x)

    # OLS regression
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()
    beta = model.params.iloc[1]
    r2 = model.rsquared

    # Cointegration
    coint_stat, coint_p, crit_vals = coint(y, x)

    print('=' * 60)
    print(f'{name} FACTOR COMPARISON: Our vs Official Ken French')
    print('=' * 60)
    print(f'  Pearson Correlation:  {corr:.4f}  (p={corr_p:.2e})')
    print(f'  Beta:                 {beta:.4f}')
    print(f'  R-squared:            {r2:.4f}')
    print(f'  Cointegration:        stat={coint_stat:.4f}, p={coint_p:.4f}')
    print(f'    Critical values:    1%={crit_vals[0]:.2f}, 5%={crit_vals[1]:.2f}, 10%={crit_vals[2]:.2f}')
    print('=' * 60)
    print()
    return {'factor': name, 'corr': corr, 'beta': beta, 'r2': r2,
            'coint_stat': coint_stat, 'coint_p': coint_p}


smb_stats = compare_factor('Our SMB', 'FF SMB', 'SMB')
hml_stats = compare_factor('Our HML', 'FF HML', 'HML')

## 14. Visual Comparison

Time-series and cumulative-return plots help us visually assess how well
our replicated factors track the official Ken French series.

In [ ]:
# ── Time series plots ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# SMB time series
axes[0, 0].plot(ff_comp['date'], ff_comp['Our SMB'], label='Our SMB', linewidth=1.5)
axes[0, 0].plot(ff_comp['date'], ff_comp['FF SMB'], label='FF SMB', linestyle='--', linewidth=1.5)
axes[0, 0].set_title('SMB: Monthly Returns', fontsize=12)
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# HML time series
axes[0, 1].plot(ff_comp['date'], ff_comp['Our HML'], label='Our HML', linewidth=1.5)
axes[0, 1].plot(ff_comp['date'], ff_comp['FF HML'], label='FF HML', linestyle='--', linewidth=1.5)
axes[0, 1].set_title('HML: Monthly Returns', fontsize=12)
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# SMB cumulative
cum_smb = (1 + ff_comp[['Our SMB', 'FF SMB']]).cumprod()
axes[1, 0].plot(ff_comp['date'], cum_smb['Our SMB'], label='Our SMB', linewidth=1.5)
axes[1, 0].plot(ff_comp['date'], cum_smb['FF SMB'], label='FF SMB', linestyle='--', linewidth=1.5)
axes[1, 0].set_title('SMB: Cumulative Returns', fontsize=12)
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# HML cumulative
cum_hml = (1 + ff_comp[['Our HML', 'FF HML']]).cumprod()
axes[1, 1].plot(ff_comp['date'], cum_hml['Our HML'], label='Our HML', linewidth=1.5)
axes[1, 1].plot(ff_comp['date'], cum_hml['FF HML'], label='FF HML', linestyle='--', linewidth=1.5)
axes[1, 1].set_title('HML: Cumulative Returns', fontsize=12)
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

fig.suptitle('Fama-French Factor Replication vs Official Ken French Factors', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter plots ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (our, ff, name) in zip(axes, [
    ('Our SMB', 'FF SMB', 'SMB'),
    ('Our HML', 'FF HML', 'HML'),
]):
    ax.scatter(ff_comp[ff], ff_comp[our], alpha=0.3, s=10)
    # 45-degree line
    lims = [min(ff_comp[ff].min(), ff_comp[our].min()),
            max(ff_comp[ff].max(), ff_comp[our].max())]
    ax.plot(lims, lims, 'r--', linewidth=1, label='45-degree line')
    ax.set_xlabel(f'Official FF {name}')
    ax.set_ylabel(f'Our {name}')
    ax.set_title(f'{name}: Our vs Official', fontsize=12)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 15. Summary Statistics

In [ ]:
# ── Summary statistics ───────────────────────────────────────────
summary_cols = ['Our SMB', 'FF SMB', 'Our HML', 'FF HML']
returns = ff_comp[summary_cols].copy()

summary = pd.DataFrame(columns=['Mean', 'Std Dev', 'Sharpe (monthly)', 'Min', 'Max'])
for col in summary_cols:
    r = returns[col].dropna()
    summary.loc[col] = [
        f'{r.mean():.4f}',
        f'{r.std():.4f}',
        f'{r.mean() / r.std():.4f}' if r.std() != 0 else 'N/A',
        f'{r.min():.4f}',
        f'{r.max():.4f}',
    ]

print('Monthly return summary statistics:')
display(summary)

## 16. Conclusion

This notebook replicates the Fama-French SMB and HML factors from raw WRDS
data following the methodology of Fama and French (1993). Key design choices include:

- **June portfolio formation** to ensure accounting data availability
- **NYSE-only breakpoints** to avoid small-stock bias
- **December market equity** in the B/M denominator for proper timing
- **Preferred stock fallback hierarchy** for robust book equity
- **Delisting return adjustment** to avoid survivorship bias
- **Permco-level ME aggregation** for correct firm-level market cap
- **Value weighting** for economically meaningful portfolio returns

### Reference

Fama, E.F. and French, K.R. (1993), "Common Risk Factors in the Returns on
Stocks and Bonds", *Journal of Financial Economics*, 33(1), 3-56.